# 20.9 回答出現正確名詞，就算看懂照片了嗎？


一張圖有「人、腳踏車」，可能是有人騎車，也可能是人站在車旁。答出這兩個名詞，還沒有回答他在做什麼。描述照片需要把看得見的物件、動作與關係接起來，再確認沒有多添畫面外的故事。

先讀[11.14](https://birdhackor.github.io/tiny-perceptron-vlm/11.14.html)的整圖關係。物件回答「有什麼」，動作回答「正在做什麼」，關係回答「在哪裡、與誰相連」。同一張圖可以問不同項目；評分應跟著問題的主要要求，不用一個名詞命中替所有題加分。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural_photo_evidence.svg")))

先用兩張手寫回答卡，隔離物件與活動的差別：


In [ ]:
truth = {"objects": {"人", "腳踏車"}, "activity": "騎車"}
cards = [
    {"objects": {"人", "腳踏車"}, "activity": "騎車"},
    {"objects": {"人", "腳踏車"}, "activity": "站在車旁"},
]
for index, card in enumerate(cards):
    print("卡", index, "物件吻合", card["objects"] == truth["objects"])
    print("活動符合本題", card["activity"] == truth["activity"])

兩張卡的物件集合都相同，活動只有第一張符合。集合不看名詞順序；本例的活動標註則由我們先寫好，沒有讓程式讀圖。正式自然語言可以用不同措辭表達同一意思，需要按可見事實判斷，不能要求每句都與單一參考描述逐字相同。

回到[20.4訓練示範](https://birdhackor.github.io/tiny-perceptron-vlm/20.4.html)中的兩隻貓：問「有幾隻」時，「兩隻」已直接完成。問「左邊那隻在做什麼」時，要核對抬前腳等姿勢；若又說牠正在等主人回家，那是照片沒有支持的意圖。回答更長，不會自動更完整。先看所問事實，再查新增細節，是讀整段回答的順序。

還可以保持問題與先前對話（歷史）相同，只換另一張圖，核對答案是否隨畫面改變。換的是實際像素，不把物件名稱偷偷塞進文字。不過少量換圖只能支持那組對照；廣泛能力仍看留出照片的分項檢查；這些照片未參與本課訓練，預留來檢查回答，摘要見[20.13](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)。

練習替「有人拿著杯子」寫一題物件問題與一題關係問題，再各寫一個畫面不支持的猜測。你已能檢查照片描述，下一節把要求收緊到圖片上的每一個字。
